# 第 5 章数据完整性 —— Drive 侧探针 + ①-c 干净 manifest 补评

**背景**：`docs/data_integrity_open_items.md` 第 ① ③ 条已核实成立；波及面评估的独立复核见
`paper/thesis_ch5/data_integrity_impact_assessment_review.md`。③ 已在本机用现存逐回合记录
零成本量化完毕，本 notebook 只处理**必须在 Drive 侧才能做**的两件事。

| 段 | 目的 | 开销 |
|---|---|---|
| §1 探针 A | 2000 / 1000 格的 ReBRAC-Q 检查点（`.pt`）是否还在 Drive | 秒级，纯 I/O |
| §2 探针 B | 含噪 2000 数据集的 `seed`；全量 `offline_data/` 污染面审计 | 分钟级，无 GPU |
| §3 ①-c | 在干净 manifest 上补评 2000 与 1000 两格 × 5 种子 | 10 次纯评估，无训练 |
| §4 回传 | 该同步回 git 的小文件清单 | — |

**先只跑 §1 §2，看结果再决定是否放开 §3**（§3 由 `RUN_CLEAN_PROBE` 开关控制，默认 `False`）。

**不训练、不重采数据、不动任何 `.tex`。**

## 0. 挂载 Drive + cd 进项目

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd drive/"MyDrive"/"Colab Notebooks"/"new_offRL"/"rl_v2_5"
!pwd
!git log --oneline -3

/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5
/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5
fatal: not a git repository (or any parent up to mount point /content)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


## 1. 探针 A —— 2000 / 1000 格的检查点还在吗

这一条决定处置格局：**检查点在 → ①-c（补评，测出污染幅度）成立；已清 → 只剩「重采重跑」与
「保留数字 + 降级翻转论述」二选一。**

本机（Windows 副机）`results/offline/**` 下 `.pt` 计数为 **0**，选点记录 `selected_checkpoint.json`
里只留了文件名（如 `agent_step_00066752.pt`）。下面既查目录存在性，也查**被选中的那个** `.pt`
是否还在——目录里有一堆 `.pt` 但恰好缺选中那个，等于没有。

In [3]:
import json, glob
from pathlib import Path

CELLS = {
    "cross-1000": "crosscomp_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep1000",
    "cross-2000": "crosscomp_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep2000",
}
PAIR   = "actorb_4p0__criticb_2p0"
SEEDS  = [42, 43, 44, 45, 46]
CKPT_ROOT = Path("checkpoints/offline/rebrac/formal")
RES_ROOT  = Path("results/offline/rebrac/formal")

PROBE_A = {}
print(f"{'cell':11s} {'seed':>4s} {'run_dir':>8s} {'#.pt':>5s} {'selected agent_file':32s} {'selected .pt':>12s}")
print("-" * 82)
for label, ds in CELLS.items():
    for seed in SEEDS:
        run_dir = CKPT_ROOT / ds / PAIR / f"seed_{seed}"
        sel     = RES_ROOT / ds / PAIR / "selection" / f"seed_{seed}" / "selected_checkpoint.json"
        agent_file = None
        if sel.exists():
            agent_file = json.loads(sel.read_text(encoding="utf-8"))["best"]["agent_file"]
        n_pt = len(list(run_dir.glob("*.pt"))) if run_dir.is_dir() else 0
        ok   = bool(agent_file) and (run_dir / agent_file).exists()
        PROBE_A[(label, seed)] = {"run_dir": str(run_dir), "agent_file": agent_file, "ok": ok}
        print(f"{label:11s} {seed:4d} {str(run_dir.is_dir()):>8s} {n_pt:5d} "
              f"{str(agent_file):32s} {('YES' if ok else 'NO'):>12s}")

READY = all(v["ok"] for v in PROBE_A.values())
print()
print("=" * 82)
print("探针 A 判定:", "全部 10 个选中检查点在位 → ①-c 可做" if READY
      else "有缺失 → 见下一格的兜底搜索；仍缺则 ①-c 作废")

cell        seed  run_dir  #.pt selected agent_file              selected .pt
----------------------------------------------------------------------------------
cross-1000    42     True    10 agent_step_00033432.pt                    YES
cross-1000    43     True    10 agent_step_00033432.pt                    YES
cross-1000    44     True    10 agent_step_00033432.pt                    YES
cross-1000    45     True    10 agent_final.pt                            YES
cross-1000    46     True    10 agent_step_00028656.pt                    YES
cross-2000    42     True    10 agent_step_00066752.pt                    YES
cross-2000    43     True    10 agent_step_00066752.pt                    YES
cross-2000    44     True    10 agent_step_00057216.pt                    YES
cross-2000    45     True    10 agent_step_00066752.pt                    YES
cross-2000    46     True    10 agent_step_00066752.pt                    YES

探针 A 判定: 全部 10 个选中检查点在位 → ①-c 可做


In [4]:
# 兜底：若上面路径落空，说明 Drive 侧目录布局与预期不同，在 checkpoints/ 下全局搜一遍
if not READY:
    print("checkpoints/ 下所有含 ep1000 / ep2000 的 seed 目录：")
    for p in sorted(glob.glob("checkpoints/**/*_ep[12]000/**/seed_*", recursive=True))[:80]:
        n = len(list(Path(p).glob("*.pt")))
        print(f"  {n:3d} .pt   {p}")
    print()
    print("checkpoints/ 顶层：")
    for p in sorted(glob.glob("checkpoints/*")):
        print("  ", p)
else:
    print("（探针 A 已通过，跳过兜底搜索）")

（探针 A 已通过，跳过兜底搜索）


## 2. 探针 B —— 含噪 2000 数据集 + 全量污染面审计

复核发现的漏项：§5.6.2 的 noisy-support 诊断有一半坐在 **2000 回合**数据上，其中含噪 2000
数据集（`crosscomp_..._noise0p05clip0p15_ep2000`）**从未进过污染面枚举**——之前那句「10 个数据集
只有一个受影响」数的是本机 `offline_data/` 目录，而该数据集只在 Drive。

`collect_offline_data.py` 的 `--seed` 默认为 `0`，本机现存 10 个数据集无一例外都是 0。若含噪 2000
沿用同一调用，其训练种子区间同为 0–1999，**完整包含评估 manifest 的 1250–1349**。

In [5]:
import json
from pathlib import Path

print(f"{'dataset':78s} {'seed':>5s} {'episodes':>9s} {'succ':>6s}  风险")
print("-" * 108)
MIN_MANIFEST_SEED = 1100   # 全部 8 个 benchmark key 中最小的 manifest_seed
for meta_path in sorted(Path("offline_data").glob("*/metadata.json")):
    m = json.loads(meta_path.read_text(encoding="utf-8"))
    seed, eps = m.get("seed"), m.get("num_episodes")
    risky = (seed is not None and eps is not None
             and seed <= MIN_MANIFEST_SEED < seed + eps)
    print(f"{meta_path.parent.name:78s} {str(seed):>5s} {str(eps):>9s} "
          f"{str(m.get('success_rate')):>6s}  {'⚠ 可能相交' if risky else 'ok'}")

dataset                                                                         seed  episodes   succ  风险
------------------------------------------------------------------------------------------------------------
crosscomp_s0_h4_arrival_v2_re150_u10cross_fixdone_ep1000                           0      1000   0.87  ok
crosscomp_s0_h4_arrival_v2_simple_re150_u10upstream_fixdone_ep1000                 0      1000    1.0  ok
crosscomp_s0_h4_efficiency_v2_re150_u10cross                                       0       500  0.864  ok
crosscomp_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep1000                        0      1000   0.87  ok
crosscomp_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep2000                        0      2000 0.8855  ⚠ 可能相交
crosscomp_s0_h4_efficiency_v2_re150_u10upstream_fixdone_ep1000                     0      1000    1.0  ok
crosscomp_s0_h4_efficiency_v2_re150tandem_u10cross_fixdone_ep1000                  0      1000  0.964  ok
crosscomp_s0_h4_effv2_re150_u10cross   

In [7]:
# 权威判定：range pass 覆盖 Drive 侧全部数据集 × 全部 manifest（只比对同流场/几何/目标速度的组合）
!python -m scripts.audit_seed_overlap

datasets: 24   manifests: 22

[clean  ] crosscomp_s0_h4_arrival_v2_re150_u10cross_fixdone_ep1000  seeds 0..999
[clean  ] crosscomp_s0_h4_arrival_v2_simple_re150_u10upstream_fixdone_ep1000  seeds 0..999
[clean  ] crosscomp_s0_h4_efficiency_v2_re150_u10cross  seeds 0..499
[clean  ] crosscomp_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep1000  seeds 0..999
[OVERLAP] crosscomp_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep2000  seeds 0..1999
    OVERLAP 100/100 with offline_rebrac_broad/test_100/single_u10_cross_tgt15.json (seeds 1250..1349)
    OVERLAP 40/40 with offline_rebrac_broad/val_40/single_u10_cross_tgt15.json (seeds 1250..1289)
    OVERLAP 100/100 with offline_rebrac_screen/test_100/single_u10_cross_tgt15.json (seeds 1250..1349)
    OVERLAP 40/40 with offline_rebrac_screen/val_40/single_u10_cross_tgt15.json (seeds 1250..1289)
    OVERLAP 40/40 with offline_rebrac_worldcomp_epoch_probe/test_40/single_u10_cross_tgt15.json (seeds 1250..1289)
    OVERLAP 40/40 with offline_rebrac_world

In [9]:
# 若上一格报出含噪 2000 相交，用实例级比对钉死（重放 reset RNG，不跑仿真）
NOISY_2000 = "crosscomp_s0_h4_efficiency_v2_re150_u10cross_fixdone_noise0p05clip0p15_ep2000"
import os
if os.path.isdir(f"offline_data/{NOISY_2000}"):
    !python -m scripts.audit_seed_overlap --verify {NOISY_2000} single_u10_cross_tgt15_ep100
else:
    print(f"{NOISY_2000} 不在 Drive 侧 offline_data/ 下；若确已删除，则该读数无法再复核，")
    print("§5.6.2 的 noisy-support 段只能走披露路线。")

dataset : crosscomp_s0_h4_efficiency_v2_re150_u10cross_fixdone_noise0p05clip0p15_ep2000  seeds 0..1999
manifest: single_u10_cross_tgt15_ep100.json  100 episodes
seeds in both ranges     : 100/100
identical task instances : 100/100


## 3. ①-c —— 干净 manifest 补评（**需 §1 通过后再放开**）

只有在**训练区间 0–1999** 与**现测试集 1250–1349** 都不相交的评估集上，才能测出 ① 的污染幅度 δ。

- 配方本机已验证：`scripts/generate_benchmark_manifest.py --seed 1250` 逐条重现现有
  `benchmarks/single_u10_cross_tgt15_ep100.json`（100/100，`flow_time`/`start_xy`/`goal_xy`/
  `initial_heading` 全等）→ 换 `--seed 3000` 即得等价但不相交的评估集。**零改代码。**
- **1000 与 2000 两格必须同 manifest** —— 翻转是两格之差，只补 2000 而拿 1000 的旧数字比是混口径。
- 只评估、不训练：`scripts.evaluate_offline` 从 `trainer_state.json` 读回全部协议设定。

In [10]:
RUN_CLEAN_PROBE = False   # ← §1 判定 READY 后改成 True
CLEAN_SEED      = 3000    # 训练 0..1999 / 现测试集 1250..1349 之外

In [11]:
import subprocess
from pathlib import Path

FLOW = "wake_data/wake_v8_U1p00_Re150_D12p00_dx0p60_Ti5pct_1200f_roi.npy"
CLEAN_MANIFEST = Path(f"benchmarks/clean_probe/single_u10_cross_tgt15_ep100_s{CLEAN_SEED}.json")
REPRO_MANIFEST = Path("benchmarks/clean_probe/_repro_check_s1250.json")

def gen_manifest(seed, out):
    out.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "python", "-m", "scripts.generate_benchmark_manifest",
        "--flow", FLOW,
        "--task-geometry", "cross_stream",
        "--target-speed", "1.5",
        "--initial-speed", "0.3",
        "--episodes", "100",
        "--seed", str(seed),
        "--output", str(out),
    ], check=True)

if RUN_CLEAN_PROBE:
    # 3a. 自检：同种子必须逐条重现现有 manifest，否则配方与原始生成环境不一致，后面全部作废
    gen_manifest(1250, REPRO_MANIFEST)
    a = json.loads(Path("benchmarks/single_u10_cross_tgt15_ep100.json").read_text())["episodes"]
    b = json.loads(REPRO_MANIFEST.read_text())["episodes"]
    KEYS = ["flow_time", "start_xy", "goal_xy", "initial_heading"]
    def same(x, y):
        if x["seed"] != y["seed"]:
            return False
        for k in KEYS:
            u, v = x["reset_options"][k], y["reset_options"][k]
            if isinstance(u, list):
                if any(abs(p - q) > 1e-9 for p, q in zip(u, v)):
                    return False
            elif abs(u - v) > 1e-9:
                return False
        return True
    n_same = sum(same(x, y) for x, y in zip(a, b))
    print(f"配方自检：{n_same}/100 逐条相同")
    assert n_same == 100, "配方与原始 manifest 不一致 —— 停下，不要继续补评"

    # 3b. 生成干净集
    gen_manifest(CLEAN_SEED, CLEAN_MANIFEST)
    seeds_new = {e["seed"] for e in json.loads(CLEAN_MANIFEST.read_text())["episodes"]}
    assert not (seeds_new & set(range(0, 2000))),   "与训练区间 0..1999 相交"
    assert not (seeds_new & set(range(1250, 1350))), "与现测试集相交"
    print(f"干净集 OK：{len(seeds_new)} 条，种子 {min(seeds_new)}..{max(seeds_new)}，与两个区间均不相交")
else:
    print("RUN_CLEAN_PROBE = False —— 跳过。先看 §1 §2 的结果。")

RUN_CLEAN_PROBE = False —— 跳过。先看 §1 §2 的结果。


In [12]:
OUT_ROOT = Path("results/offline/rebrac/clean_probe")

if RUN_CLEAN_PROBE:
    assert READY, "§1 未通过：选中检查点不全，不能补评"
    for (label, seed), info in PROBE_A.items():
        out_json = OUT_ROOT / label / f"seed_{seed}.json"
        if out_json.exists():
            print(f"[skip] {out_json}")
            continue
        out_json.parent.mkdir(parents=True, exist_ok=True)
        print(f"\n[eval] {label} seed {seed} — {info['agent_file']}")
        subprocess.run([
            "python", "-m", "scripts.evaluate_offline",
            "--checkpoint", info["run_dir"],
            "--agent-file", info["agent_file"],
            "--manifest", str(CLEAN_MANIFEST),
            "--output-json", str(out_json),
            "--device", "cuda",
            "--num-workers", "6",
            "--worker-device", "cpu",
        ], check=True)
    print("\n补评完成")
else:
    print("RUN_CLEAN_PROBE = False —— 跳过。")

RUN_CLEAN_PROBE = False —— 跳过。


### 3c. 读数与判定

判定门槛（依据 `paper/thesis_ch5/data_integrity_impact_assessment_review.md` §4 ★3）：

- 章内真实断言是「**未再检出**回落」，不是「2000 > 1000」；
- δ ≳ 2 pp → 「点估计反而略高」半句失效；
- δ ≳ 5–7 pp → 才构成与 TD3+BC（−7.6 pp）/ 纯 BC（−5.4 pp）同量级的回落，翻转论述才真被推翻。

已刊值：cross-1000 = 0.902 ± 0.021、cross-2000 = 0.918 ± 0.030，差 **+1.6 pp**。
剔除参与选点的 40 条后（本机已算）：0.907 / 0.937，差 **+3.0 pp**，5/5 种子不为负。

In [13]:
import math

def load_cell(label):
    vals = []
    for seed in SEEDS:
        p = OUT_ROOT / label / f"seed_{seed}.json"
        if p.exists():
            vals.append((seed, json.loads(p.read_text(encoding="utf-8"))["eval_success_rate"]))
    return vals

if RUN_CLEAN_PROBE:
    PUBLISHED = {"cross-1000": 0.902, "cross-2000": 0.918}
    means = {}
    for label in CELLS:
        vals = load_cell(label)
        if not vals:
            print(f"{label}: 无结果"); continue
        xs = [v for _, v in vals]
        m  = sum(xs) / len(xs)
        sd = math.sqrt(sum((x - m) ** 2 for x in xs) / len(xs))   # 总体口径，与章内刊值一致
        means[label] = xs
        print(f"{label}: 逐种子 {[round(x, 3) for x in xs]}  干净集 {m:.4f} ± {sd:.4f}  "
              f"已刊 {PUBLISHED[label]:.3f}  δ = {100 * (PUBLISHED[label] - m):+.1f} pp")

    if len(means) == 2:
        a, b = means["cross-2000"], means["cross-1000"]
        d = [x - y for x, y in zip(a, b)]
        md_ = sum(d) / len(d)
        sd_ = math.sqrt(sum((x - md_) ** 2 for x in d) / (len(d) - 1))
        se  = sd_ / math.sqrt(len(d))
        print()
        print(f"干净集上的翻转：{sum(a)/len(a):.4f} 对 {sum(b)/len(b):.4f} = {100*md_:+.1f} pp")
        print(f"  配对 t = {md_/se:+.3f}，95% CI = [{100*(md_-2.776*se):+.1f}, {100*(md_+2.776*se):+.1f}] pp")
        print(f"  逐种子差 {[round(x, 3) for x in d]}，其中不为负 {sum(x >= 0 for x in d)}/5")
        print()
        print("对照：已刊 100 回合 +1.6 pp（t=1.06，1 负种子）；剔除选点 40 条后 +3.0 pp（t=2.09，5/5 不为负）")
else:
    print("RUN_CLEAN_PROBE = False —— 跳过。")

RUN_CLEAN_PROBE = False —— 跳过。


## 4. 回传 git 的清单

只回传小文件，**不回传 `.pt`**：

- `benchmarks/clean_probe/single_u10_cross_tgt15_ep100_s3000.json`（干净评估集，可复现凭据）
- `results/offline/rebrac/clean_probe/**/seed_*.json`（10 份补评结果，含逐回合记录）
- 本 notebook 的 `_completed` 版本

`benchmarks/clean_probe/_repro_check_s1250.json` 是一次性自检产物，**不回传**。

回传后在本机：把读数写进 `docs/data_integrity_open_items.md` 第 ① 条，并按
`paper/thesis_ch5/data_integrity_impact_assessment_review.md` §5 的 P4 把 ① 与 ③ 合成**一批**
整改（重流程只付一次）。**在那之前不要动任何 `.tex`。**